# Transposed Convolutions 

Transposed convolutions are not a mathematical inverse of the standard convolution operation. If we have a 2d convolution, we can unroll this convolution into a standard matrix multiplication operation. If we have the given image $x$ and flatten out the shape into a 1d column vector we can create a sparse matrix $C$ (this is known as a doubly block Toeplitz matrix).

A Toeplitz matrix can be shown with the example.

$$
\begin{bmatrix}
a & b & c & d & e \\
f & a & b & c & d \\
g & f & a & b & c \\
h & g & f & a & b \\
i & h & g & f & a
\end{bmatrix}
$$

Now standard Toeplitz matrcies are used for 1D signals, but since images use two spatial dimensions (height and width), we'll use a doubly block toeplitz (DBT) matrix. 

The DBT will map a 2D convolution into a 1D matrix multiplication $(y=Cx)$. It scales up the 1D shifting properties of a standard Toeplitz matrix to handle a 2D grid. The jump from a Toeplitz to DBT isn't complicated though, as the only difference is that instead of having a single `value` for each `element` in the matrix, we'll fit entire *sub-matrices* $(A, B, C)$ along the diagonals:


$$
\begin{bmatrix}
t_0 & t_{-1} & t_{-2} & t_{-3} \\
t_1 & t_0 & t_{-1} & t_{-2} \\
t_2 & t_1 & t_0 & t_{-1} \\
t_3 & t_2 & t_1 & t_0
\end{bmatrix}
$$


$$
\begin{bmatrix}
\begin{bmatrix} a_0 & a_{-1} & a_{-2} \\ a_1 & a_0 & a_{-1} \\ a_2 & a_1 & a_0 \end{bmatrix} & 
\begin{bmatrix} b_0 & b_{-1} & b_{-2} \\ b_1 & b_0 & b_{-1} \\ b_2 & b_1 & b_0 \end{bmatrix} & 
\begin{bmatrix} c_0 & c_{-1} & c_{-2} \\ c_1 & c_0 & c_{-1} \\ c_2 & c_1 & c_0 \end{bmatrix} & 
\begin{bmatrix} d_0 & d_{-1} & d_{-2} \\ d_1 & d_0 & d_{-1} \\ d_2 & d_1 & d_0 \end{bmatrix} \\ \\
\begin{bmatrix} e_0 & e_{-1} & e_{-2} \\ e_1 & e_0 & e_{-1} \\ e_2 & e_1 & e_0 \end{bmatrix} & 
\begin{bmatrix} a_0 & a_{-1} & a_{-2} \\ a_1 & a_0 & a_{-1} \\ a_2 & a_1 & a_0 \end{bmatrix} & 
\begin{bmatrix} b_0 & b_{-1} & b_{-2} \\ b_1 & b_0 & b_{-1} \\ b_2 & b_1 & b_0 \end{bmatrix} & 
\begin{bmatrix} c_0 & c_{-1} & c_{-2} \\ c_1 & c_0 & c_{-1} \\ c_2 & c_1 & c_0 \end{bmatrix} \\ \\
\begin{bmatrix} f_0 & f_{-1} & f_{-2} \\ f_1 & f_0 & f_{-1} \\ f_2 & f_1 & f_0 \end{bmatrix} & 
\begin{bmatrix} e_0 & e_{-1} & e_{-2} \\ e_1 & e_0 & e_{-1} \\ e_2 & e_1 & e_0 \end{bmatrix} & 
\begin{bmatrix} a_0 & a_{-1} & a_{-2} \\ a_1 & a_0 & a_{-1} \\ a_2 & a_1 & a_0 \end{bmatrix} & 
\begin{bmatrix} b_0 & b_{-1} & b_{-2} \\ b_1 & b_0 & b_{-1} \\ b_2 & b_1 & b_0 \end{bmatrix} \\ \\
\begin{bmatrix} g_0 & g_{-1} & g_{-2} \\ g_1 & g_0 & g_{-1} \\ g_2 & g_1 & g_0 \end{bmatrix} & 
\begin{bmatrix} f_0 & f_{-1} & f_{-2} \\ f_1 & f_0 & f_{-1} \\ f_2 & f_1 & f_0 \end{bmatrix} & 
\begin{bmatrix} e_0 & e_{-1} & e_{-2} \\ e_1 & e_0 & e_{-1} \\ e_2 & e_1 & e_0 \end{bmatrix} & 
\begin{bmatrix} a_0 & a_{-1} & a_{-2} \\ a_1 & a_0 & a_{-1} \\ a_2 & a_1 & a_0 \end{bmatrix}
\end{bmatrix}
$$

### But in Relation to CNNs, What are $x$, $C^T$, and $y$? 

- Our original input image $x$ would have the shape $(H_{in} W_{in} \times 1)$. We would flatten out the dimensions of the image (based on its height and width) into the 1D vector. 

- What about $C$? This is our DBT matrix. This matrix is made entirely out of the convolutional kernel weights. If we had assigned 1 kernel with kernel sizes being 3x3. Without padding, we'll have our DBT become a sparse matrix of shape $(H_{out} W_{out} \times H_{in} W_{in})$.
    - We actually already have our input and output heights and widths in our forward pass in our code, so I won't repeat the formula here. 
- $y$ (**The output feature map vector**): This is the resulting feature map, flattened into a 1D column vector with its shape being $(H_{out}, W_{out} \times 1)$

- Why do we have to transpose $C$? When we transpose $C$, we'll get $C^T$ with its shape becomming $(H_{in} W_{in} \times H_{out} W_{out})$. We are going to need $C^T$ to compute $C^T y$. 
    - Computing $C^T y$ will have us return to our exact dimensions as the original input x. We can verify this by checking its matrix equation and its shapes. 
        $$Cx = y$$
        
        $$(H_{out}W_{out} \times H_{in}W_{in}) \times ({H_{in} W_{in} \times 1}) = (H_{out} W_{out} \times 1)$$
        - The operation is valid as we have the form $(k \times l) \times (l \times n) = (k \times n)$ where the columns of the matrix on the left correspond to the rows of the matrix on the right (in this case its a 1d column vector) and the resulting shape must mean that we have $(k \times n)$ which is represented with our *out* variables.

        - Now lets check the tranposed version (which we must confirm that the `dinputs` shape can map back to `dweights` shape or in other words we can get back to our $x$ shape starting from $C^T y$)

        $$C^T y = x$$

        $$(H_{in}W_{in} \times H_{out}W_{out}) \times ({H_{out} W_{out} \times 1}) = (H_{in} W_{in} \times 1)$$
        
        - While it is the case that we have arrived back at our input shape, we won't arrive at the exact same values as our input shapes. However, this is the whole point of using a transpose on matrix C! Doing this formula allows us to route the errors back to their spatial origins. 

### Example of our DBT in action 

Lets suppose that we have an input image $x$ with shape $3\times3$ and a $2\times2$ kernel $W$ that we want to slide over the image. 

$$
x = \begin{bmatrix} 
x_{11} & x_{12} & x_{13} \\ 
x_{21} & x_{22} & x_{23} \\ 
x_{31} & x_{32} & x_{33} 
\end{bmatrix}, \quad 
W = \begin{bmatrix} 
w_{11} & w_{12} \\ 
w_{21} & w_{22} 
\end{bmatrix}
$$

We need to make assumptions on the padding and stride of our forward convolution, in this case we can assume stride = 1 and we have no padding (valid), thus our output y will become a 2x2 matrix
$$
y = \begin{bmatrix}
y_{11} & y_{12} \\
y_{21} & y_{22} 
\end{bmatrix}
$$

Our main rule about $x$ and $y$ is that we must flatten both $x$ and $y$ into 1D column vectors by reading them row by row:

$$
x = \begin{bmatrix} 
x_{11} \\ x_{12} \\ x_{13} \\ x_{21} \\ x_{22} \\ x_{23} \\ x_{31} \\ x_{32} \\ x_{33} 
\end{bmatrix} \text{ (Shape: } 9 \times 1\text{)}, \quad 
y = \begin{bmatrix} 
y_{11} \\ y_{12} \\ y_{21} \\ y_{22} 
\end{bmatrix} \text{ (Shape: } 4 \times 1\text{)}
$$

Since we know the out and in dimension sizes, we can say that C has the shape $({2\cdot2 \times 3\cdot3})$ or shape $(4, 9)$

After learning about the shape of $C$, we can construct the actual values inside $C$. Every single row of $C$ represents a single position of the sliding window kernel over the image. Our final matrix C would look like:

$$
\begin{bmatrix} 
y_{11} \\ \\ y_{12} \\ \\ y_{21} \\ \\ y_{22} 
\end{bmatrix} = 
\begin{bmatrix}
w_{11} & w_{12} & 0 & w_{21} & w_{22} & 0 & 0 & 0 & 0 \\
0 & w_{11} & w_{12} & 0 & w_{21} & w_{22} & 0 & 0 & 0 \\
0 & 0 & 0 & w_{11} & w_{12} & 0 & w_{21} & w_{22} & 0 \\
0 & 0 & 0 & 0 & w_{11} & w_{12} & 0 & w_{21} & w_{22}
\end{bmatrix}
\begin{bmatrix} 
x_{11} \\ x_{12} \\ x_{13} \\ x_{21} \\ x_{22} \\ x_{23} \\ x_{31} \\ x_{32} \\ x_{33} 
\end{bmatrix}
$$

How is this a "Doubly Block" Toeplitz Matrix? 

If we look at each row of the kernel, there are three "blocks" created by the rows of the kernel:
$$
A = \begin{bmatrix} w_{11} & w_{12} & 0 \end{bmatrix}, \quad 
B = \begin{bmatrix} w_{21} & w_{22} & 0 \end{bmatrix}, \quad 
0 = \begin{bmatrix} 0 & 0 & 0 \end{bmatrix}
$$

* Block A corresponds to the first row of the kernel plus a zero padding slot because the image width is 3. 
    * We know $w_{11}$ and $w_{12}$ are the first row of the $2\times2$ kernel. 
    * The trailing 0 is used for padding. It spans the 3rd pixel of the current image row, which ensures that the kernel doesn't warp around and multiply the edge of the image. 
    * Block 0 exists to skip entire rows of the image that the kernel isn't currently touching. For example, on the first iteration where we'd perform the first convolution (we map this to $y_{11}$), we shouldn't look at Row 3 $(x_{31}​,x_{32}​,x_{33}​)$ at all. 

# Backpropagation Example for Using Our DBT $C$

During the backwards pass, we'll recieve a 1D vector representing the sensitivity of the Loss wrt. the output $y$. Since $y$ will have shape $(H_{out}W_{out}\times 1)$ or in our example from above $(4\times1)$ column vector:

$$\frac{\partial L}{\partial y} = 
\begin{bmatrix} 
\partial_{11} \\ \\ \partial_{12} \\ \\ \partial_{21} \\ \\ \partial_{22} 
\end{bmatrix}
$$

The goal of the backwards pass is to compute the gradient vector $\frac{\partial L }{\partial x}$ of shape $(H_{in}W_{in}\times1)$ or $(9\times1)$. The chain rule dictates:

$$
\frac{\partial L}{\partial x}=(\frac{\partial y}{\partial x})^T \frac{\partial L}{\partial y}
$$

### Constructing the Jacobian Matrix $\frac{\partial y}{\partial x}$


The Jacobian matrix will map how every single output element (rows) will change wrt. every single input element (columns). We have 4 output elements and 9 input element, so the Jacobian is a $(4\times9)$ matrix:

$$
\frac{\partial y}{\partial x} = \begin{bmatrix}
\frac{\partial y_{11}}{\partial x_{11}} & \frac{\partial y_{11}}{\partial x_{12}} & \frac{\partial y_{11}}{\partial x_{13}} & \frac{\partial y_{11}}{\partial x_{21}} & \frac{\partial y_{11}}{\partial x_{22}} & \frac{\partial y_{11}}{\partial x_{23}} & \frac{\partial y_{11}}{\partial x_{31}} & \frac{\partial y_{11}}{\partial x_{32}} & \frac{\partial y_{11}}{\partial x_{33}} \\ \\
\frac{\partial y_{12}}{\partial x_{11}} & \frac{\partial y_{12}}{\partial x_{12}} & \frac{\partial y_{12}}{\partial x_{13}} & \frac{\partial y_{12}}{\partial x_{21}} & \frac{\partial y_{12}}{\partial x_{22}} & \frac{\partial y_{12}}{\partial x_{23}} & \frac{\partial y_{12}}{\partial x_{31}} & \frac{\partial y_{12}}{\partial x_{32}} & \frac{\partial y_{12}}{\partial x_{33}} \\ \\
\frac{\partial y_{21}}{\partial x_{11}} & \frac{\partial y_{21}}{\partial x_{12}} & \frac{\partial y_{21}}{\partial x_{13}} & \frac{\partial y_{21}}{\partial x_{21}} & \frac{\partial y_{21}}{\partial x_{22}} & \frac{\partial y_{21}}{\partial x_{23}} & \frac{\partial y_{21}}{\partial x_{31}} & \frac{\partial y_{21}}{\partial x_{32}} & \frac{\partial y_{21}}{\partial x_{33}} \\ \\
\frac{\partial y_{22}}{\partial x_{11}} & \frac{\partial y_{22}}{\partial x_{12}} & \frac{\partial y_{22}}{\partial x_{13}} & \frac{\partial y_{22}}{\partial x_{21}} & \frac{\partial y_{22}}{\partial x_{22}} & \frac{\partial y_{22}}{\partial x_{23}} & \frac{\partial y_{22}}{\partial x_{31}} & \frac{\partial y_{22}}{\partial x_{32}} & \frac{\partial y_{22}}{\partial x_{33}}
\end{bmatrix}
$$

When looking back at the forward pass for the four output elements, we realize that each row element corresponds to which partial sneed to be calculated. In our case, each padding block 0 can be ignored, as this doesn't effect the outcome of output y, and each padding added in the block can also technically be ignored too. Therefore, for the weight entries that are inside the DBT matrix $C$ we just need to transpose our original matrix $C$ to move on to the next step. 

$$
\left(\frac{\partial y}{\partial x}\right)^T = C^T = \begin{bmatrix}
w_{11} & 0 & 0 & 0 \\
w_{12} & w_{11} & 0 & 0 \\
0 & w_{12} & 0 & 0 \\
w_{21} & 0 & w_{11} & 0 \\
w_{22} & w_{21} & w_{12} & w_{11} \\
0 & w_{22} & 0 & w_{12} \\
0 & 0 & w_{21} & 0 \\
0 & 0 & w_{22} & w_{21} \\
0 & 0 & 0 & w_{22}
\end{bmatrix} \quad \text{Shape: } (9 \times 4)
$$

### Step 3: Computing Downstream Gradients ($C^T \frac{\partial L}{\partial y}$)

We can now plug in the transposed Jacobian and see how each error accumulates for each original pixel in the $3\times3$ input image $x$:

$$
\frac{\partial L}{\partial x} = \begin{bmatrix} 
\frac{\partial L}{\partial x_{11}} \\ \frac{\partial L}{\partial x_{12}} \\ \frac{\partial L}{\partial x_{13}} \\ 
\frac{\partial L}{\partial x_{21}} \\ \frac{\partial L}{\partial x_{22}} \\ \frac{\partial L}{\partial x_{23}} \\ 
\frac{\partial L}{\partial x_{31}} \\ \frac{\partial L}{\partial x_{32}} \\ \frac{\partial L}{\partial x_{33}} 
\end{bmatrix} = \begin{bmatrix}
w_{11} & 0 & 0 & 0 \\
w_{12} & w_{11} & 0 & 0 \\
0 & w_{12} & 0 & 0 \\
w_{21} & 0 & w_{11} & 0 \\
w_{22} & w_{21} & w_{12} & w_{11} \\
0 & w_{22} & 0 & w_{12} \\
0 & 0 & w_{21} & 0 \\
0 & 0 & w_{22} & w_{21} \\
0 & 0 & 0 & w_{22}
\end{bmatrix}
\begin{bmatrix} \delta_{11} \\ \delta_{12} \\ \delta_{21} \\ \delta_{22} \end{bmatrix}
$$

Lets manually find out the algebraic formulation for some of the input pixels in $x$:

$$
\frac{\partial L}{\partial x_{11}} = w_{11} \partial_{11}
$$

* In english, the first pixel of our input image only ever influcened the first pixel on the output feature map $y$. That's why when taking the downstream gradient we only have to plug in the original weight that influenced our output, and the output itself. 

$$
\frac{\partial L}{\partial x_{12}} = w_{12} \partial_{11} + w_{11} \partial_{12}
$$

* In english, the second pixel of our input image influences the first two pixels on the output feature map. After sliding once to calculate the output element $y_{12}$ we have to shift our $w_{11}$ from $x_{11}$ to $x_{12}$, which is why we end up with the final equation above. 

$$
\begin{aligned}
\frac{\partial L}{\partial x_{13}} &= w_{12}\delta_{12} \\
\frac{\partial L}{\partial x_{21}} &= w_{21}\delta_{11} + w_{11}\delta_{21} \\
\frac{\partial L}{\partial x_{22}} &= w_{22}\delta_{11} + w_{21}\delta_{12} + w_{12}\delta_{21} + w_{11}\delta_{22} \\
\frac{\partial L}{\partial x_{23}} &= w_{22}\delta_{12} + w_{12}\delta_{22} \\
\frac{\partial L}{\partial x_{31}} &= w_{21}\delta_{21} \\
\frac{\partial L}{\partial x_{32}} &= w_{22}\delta_{21} + w_{21}\delta_{22} \\
\frac{\partial L}{\partial x_{33}} &= w_{22}\delta_{22}
\end{aligned}
$$



# Calculating the gradients using stride $S\ne 1$

In the example we assumed that $S = 1$, but what would happen if we used a stride of $S = 2$?

* In the forward pass, a larger stride will compress the output dimensions. If we don't account for this change then the downstream gradient will continue to compress across training steps. 

* To account for this, our goal is to propagate the error back like normal but still end up with the correct shapes for the downstream gradient $\frac{\partial L}{\partial x}$. Spatially, we'll have to **dialate** the upstream gradient by inserting $S-1$ rows and columns of internal zeros between every single pixel of $\frac{\partial L}{\partial y}$ before sliding the flipped kernel over it. 

## Why a Forward Stride of 2 Implies Gradient Dialation 

Suppose the forward pass still starts with a $3\times3$ image $x$ and the $2\times2$ kernel $W$. 
If we changed the forward stride to $S=2$ (assuming valid padding), the sliding window can fit once horizontally and once vertically. 
We'll arrive at an output $y=\begin{bmatrix} y_{11} \end{bmatrix}$, or a single $1\times1$ pixel.  

During backpropagation, the upstream gradient $\frac{\partial L}{\partial y} would arrive as a $1\times1$ matrix:

$$
\frac{\partial L}{\partial y} = \begin{bmatrix} \partial_{11} \end{bmatrix}
$$

If we slid the $2\times2$ flipped kernel over our $1\times1$ gradient normally, we would produce an output size of $2\times2$. However, the original input image had a size of $3\times3$ meaning that our shapes do **NOT** match. 

### Addressing the Matrix Reality of Stride 2

The DBT matrix 
$$
C_{stride =2} = \begin{bmatrix}
w_{11} & w_{12} & 0 & w_{21} & w_{22} & 0 & 0 & 0 & 0
\end{bmatrix}
$$
will have to be transposed to compute the downstream gradient ($C^T \frac{\partial L}{\partial y}$) giving a $(9\times1)$ matrix:

$$
\frac{\partial L}{\partial x} = C^T \frac{\partial L}{\partial y} = 
\begin{bmatrix} 
w_{11} \\ w_{12} \\ 0 \\ w_{21} \\ w_{22} \\ 0 \\ 0 \\ 0 \\ 0 
\end{bmatrix} 
\begin{bmatrix} \delta_{11} \end{bmatrix}
 = 
\begin{bmatrix} 
w_{11}\delta_{11} \\ w_{12}\delta_{11} \\ 0 \\ w_{21}\delta_{11} \\ w_{22}\delta_{11} \\ 0 \\ 0 \\ 0 \\ 0 
\end{bmatrix}
$$

#### Turning the Transpose Back into a Sliding Window 

If we tried to unflatten the downstream gradient back into the original $3\times3$ spaital layout we can see how the errors are distributed. 

$$
\frac{\partial L}{\partial x} = \begin{bmatrix}
w_{11}\delta_{11} & w_{12}\delta_{11} & 0 \\
w_{21}\delta_{11} & w_{22}\delta_{11} & 0 \\
0 & 0 & 0
\end{bmatrix}
$$

This is the output that we would like to find ourselves back at, but to get this exact $3\times3$ output, we have to mimic the skipped portions by dialating the gradient array.  

Since $S=2$, we insert $S-1= 1$ row and column of internal zeros between every pixel. As $\frac{\partial L}{\partial y}$ only has one pixel ($\partial_{11}$), the dialation won't change the interior values, but the outer padding forces the flipped kernel to distribute the weights over a wider spatial area. 

If $\frac{\partial L}{\partial y}$ had been larger (e.g., $2\times2$), dialiting it with internal zeros would expand it out like a checkerboard:

$$
\frac{\partial L}{\partial y} = \begin{bmatrix} 
\delta_{11} & \delta_{12} \\ 
\delta_{21} & \delta_{22} 
\end{bmatrix}
$$

$$
\delta_{\text{dilated}} = \begin{bmatrix} 
\delta_{11} & 0 & \delta_{12} \\ 
0 & 0 & 0 \\ 
\delta_{21} & 0 & \delta_{22} 
\end{bmatrix}
$$

Recall the flipped kernel $W_{\text{flipped}}$ (rotated 180 degrees):

$$
W_{\text{flipped}} = \begin{bmatrix} w_{22} & w_{21} \\ w_{12} & w_{11} \end{bmatrix}
$$

To reconstruct the downstream gradient $\frac{\partial L}{\partial x}$, we apply a standard convolution (with a backward stride $S_{\text{backward}} = 1$) over $\delta_{\text{dilated}}$ using full padding (padding of $K-1 = 1$ on all sides) to allow the kernel to cleanly step over the boundaries.

* **Top-Left Corner (Row 1, Col 1):**
  The kernel hovers such that only its bottom-right element hits the top-left element of the dilated gradient.
  $$
  \begin{bmatrix} \cdot & \cdot \\ \cdot & w_{11} \end{bmatrix} \cdot \begin{bmatrix} \cdot & \cdot \\ \cdot & \delta_{11} \end{bmatrix} \implies \frac{\partial L}{\partial x_{11}} = w_{11}\delta_{11}
  $$
* **Shift Right by 1 Pixel (Row 1, Col 2):**
  The kernel shifts right. Now, the bottom-left and bottom-right weights line up with the top row of our dilated matrix.
  $$
  \begin{bmatrix} \cdot & \cdot \\ w_{12} & w_{11} \end{bmatrix} \cdot \begin{bmatrix} \cdot & \cdot \\ \delta_{11} & 0 \end{bmatrix} \implies \frac{\partial L}{\partial x_{12}} = w_{12}\delta_{11} + w_{11}(0) = w_{12}\delta_{11}
  $$

* **Shift Right by 2 Pixels (Row 1, Col 3):**
  The kernel moves over the internal zero column to encounter the next real gradient element, $\delta_{12}$.
  $$
  \begin{bmatrix} \cdot & \cdot \\ w_{12} & w_{11} \end{bmatrix} \cdot \begin{bmatrix} \cdot & \cdot \\ 0 & \delta_{12} \end{bmatrix} \implies \frac{\partial L}{\partial x_{13}} = w_{12}(0) + w_{11}\delta_{12} = w_{11}\delta_{12}
  $$

* **Shift Right by 3 Pixels (Row 1, Col 4):**
  The kernel passes $\delta_{12}$.
  $$
  \begin{bmatrix} \cdot & \cdot \\ w_{12} & \cdot \end{bmatrix} \cdot \begin{bmatrix} \cdot & \cdot \\ \delta_{12} & \cdot \end{bmatrix} \implies \frac{\partial L}{\partial x_{14}} = w_{12}\delta_{12}
  $$

  